# Camada Silver: custos

In [0]:
from pyspark.sql import functions as F, Window

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

## Funções auxiliares

In [0]:
TOKENS_NULOS = ["NULL", "NONE", "NAN", "N/A"]      # textos que na prática significam "sem valor"


def texto(c):
    """Coluna como string, sem espaços nas pontas."""
    return F.trim(F.col(c).cast("string"))


def eh_faltante(c):
    """Null de verdade, string vazia/só espaços, ou texto tipo 'NULL'."""
    return F.col(c).isNull() | (texto(c) == "") | F.upper(texto(c)).isin(TOKENS_NULOS)


def somente_digitos(c):
    """Mantém só 0-9, como string (mesmo padrão dos ids da tb_chamados). Sem nenhum dígito -> null."""
    d = F.regexp_replace(F.col(c).cast("string"), r"[^0-9]", "")
    return F.when(d == "", F.lit(None).cast("string")).otherwise(d)


def ultima_versao(df, chaves):
    """Fica com a carga mais recente de cada chave (a Bronze é append) e descarta ingestion_datetime.
    Também elimina cópias exatas da mesma chave dentro de uma carga."""
    w = Window.partitionBy(*chaves).orderBy(F.col("ingestion_datetime").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "ingestion_datetime")


def checar(regra, df_falhas):
    """Para o notebook se alguma linha violar a regra."""
    n = df_falhas.count()
    assert n == 0, f"FALHA na validação '{regra}': {n} linha(s)"
    print(f"OK: {regra}")


def salvar(df, nome):
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{silver_schema}.{nome}"))
    print(f"{silver_schema}.{nome}: {spark.table(f'{silver_schema}.{nome}').count()} linhas")


def custo_decimal(c):
    """Texto de custo -> decimal(10,2). Tira 'R$', 'reais' e espaços e troca vírgula por ponto.
    Cobre os 4 formatos vistos na Bronze: '12,24', '0.46', 'R$ 1,46', '11.14reais'.
    Formato não reconhecido vira null (a validação da seção 3 barra o notebook se isso acontecer)."""
    limpo = F.regexp_replace(F.regexp_replace(texto(c), r"R\$|reais|\s", ""), ",", ".")
    return F.when(limpo.rlike(r"^-?\d+(\.\d+)?$"), limpo.cast("decimal(10,2)"))

## 1. Leitura da Bronze

In [0]:
b_custos = spark.table(f"{bronze_schema}.custos")
total_bronze = b_custos.count()
print(f"Linhas na Bronze: {total_bronze}")
display(b_custos)

## 2. Deduplicação
Na Bronze havia 781 cópias exatas, sem versões divergentes. A `ultima_versao` por `id_custo` remove essas cópias e também cargas repetidas.

In [0]:
custos = ultima_versao(
    b_custos.withColumn("id_custo", somente_digitos("id_custo"))
            .withColumn("id_chamado", somente_digitos("id_chamado")),
    ["id_custo"],
)
print(f"Linhas removidas na deduplicação: {total_bronze - custos.count()}")

## 3. Conversão do custo
Custos negativos viram null, conforme as conclusões da Bronze. Custos zerados são mantidos.

In [0]:
custos = custos.withColumn("_custo_num", custo_decimal("custo"))

checar("custo: todo valor preenchido na Bronze foi reconhecido como número",
       custos.filter(~eh_faltante("custo") & F.col("_custo_num").isNull()))

n_negativos = custos.filter(F.col("_custo_num") < 0).count()
print(f"Custos negativos convertidos para null: {n_negativos}")

custos = (
    custos
    .withColumn("custo", F.when(F.col("_custo_num") >= 0, F.col("_custo_num")))
    .select("id_custo", "id_chamado", "custo")
    .orderBy(F.col("id_custo").cast("long"))
)
display(custos)

## 4. Validações
A checagem de integridade usa a `silver.tb_chamados`, então a Silver de chamados precisa rodar antes deste notebook.

In [0]:
checar("custos: id_custo e id_chamado preenchidos",
       custos.filter(F.col("id_custo").isNull() | F.col("id_chamado").isNull()))
checar("custos: id_custo único", custos.groupBy("id_custo").count().filter("count > 1"))
checar("custos: no máximo um custo por chamado", custos.groupBy("id_chamado").count().filter("count > 1"))
checar("custos: custo nulo ou maior ou igual a zero", custos.filter(F.col("custo") < 0))

ids_chamados = spark.table(f"{silver_schema}.tb_chamados").select("id_chamado").distinct()
checar("custos: todo id_chamado existe na tb_chamados",
       custos.join(ids_chamados, on="id_chamado", how="left_anti"))

## 5. Gravação da Silver
Só chega aqui se todas as validações passaram.

In [0]:
salvar(custos, "tb_custos")